In [1]:
import pandas as pd
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import plotly.express as px

#these are feiner naming conventions

# datafr = datafr.rename(columns={"Time": 'Time Stamp',
# 'pCO2 (mmhg)': 'pCO2_mmHg_',
# 'pO2 (mmHg)':'pO2_mmHg_',
# 'sO2 (%)': 'sO2_%_',
# 'COHb (%)':'COHB_%_',
# 'MetHb (%)':'MetHb_%_',
# 'tHb (g/dL)':'tHb_g_dL_',
# 'Accession number':'UPI',})


 qulaity checks
 - there is a missing accession number
 - ? there are duplicate 'patient IDs' between the two files

issues to deal with
- how many ABG files? 2? 3? maybe just one?

ingest multiple CSVs

delete columns not needed

 input/select the session for each patient
 maybe query the 
- 1 use the UPI to query the patient redcap database for the patient name, and 
- 2 use the UPI to query the session database for the associated sessions and dates

for each upi, create a dropdown selectbox of sessions and dates


In [2]:
df1 = pd.read_csv('c1.csv', encoding='cp1252')
df2 = pd.read_csv('c2.csv', encoding='cp1252')

In [3]:
# the csv file itself displays the trailing 0s but the 0s are trimmed already in df1 and df2 
def check_df_duplicate(df_name):
    alist = []
    for i in range(0, len(df_name['Patient ID'])):    
        for j in range(i+1, len(df_name['Patient ID'])):    
            if(df_name['Patient ID'][i] == df_name['Patient ID'][j]):    
                alist.append(df_name['Patient ID'][j])
    return(alist)

check_df_duplicate(df1)
check_df_duplicate(df2)

[5.2, 4.1, 3.2, 1.2]

In [13]:
# now add the converters and the 0s are kept
df1 = pd.read_csv('c1.csv', encoding='cp1252', converters={'Patient ID': str})
df2 = pd.read_csv('c2.csv', encoding='cp1252', converters={'Patient ID': str})

check_df_duplicate(df1)
check_df_duplicate(df2)

[]

In [5]:
df_join = pd.concat([df1,df2])

In [6]:
def feinerize(datafr):
    #separate timestamp into two columns
    datafr['Time'] = pd.to_datetime(datafr['Time'])
    datafr['Date Calc'] = datafr['Time'].dt.date
    datafr['Time Calc'] =  datafr['Time'].dt.time

    #separate patient ID into two columns
    datafr[['Subject', 'Sample']] = datafr['Patient ID'].astype(str).str.split(pat='.', expand=True)

    # rename columns 
    datafr = datafr.rename(columns={"Time": 'Time Stamp',
    'pCO2 (mmHg)': 'pCO2',
    'pO2 (mmHg)':'pO2',
    'sO2 (%)': 'sO2',
    'COHb (%)':'COHb',
    'MetHb (%)':'MetHb',
    'tHb (g/dL)':'tHb',
    'Accession number':'UPI',
    'K+ (mmol/L)':'K+',
    'Na+ (mmol/L)':'Na+',
    'Ca++ (mmol/L)':'Ca++',
    'Cl- (mmol/L)':'Cl-',
    'Glu (mmol/L)': 'Glucose',
    'Lac (mmol/L)':'Lactate',
    'p50(act) (mmHg)':'p50',
    'cBase(Ecf) (mmol/L)':'cBase'})

    return datafr[allcols]

In [7]:
#fcols = feiner columns that he uses for processing
fcols = ['Time Stamp', 'Date Calc', 'Time Calc', 'Subject', 'Sample', 'Patient ID', 'pH','pCO2', 'pO2', 'sO2','COHb','MetHb','tHb','UPI']

# allcols = the rest of the columns we actually want + feiner cols
allcols = fcols + ['K+', 'Na+','Ca++','Cl-','Glucose','Lactate','p50','cBase']

In [8]:
df_join = feinerize(df_join)

/var/folders/06/_cwvk2w107zcyfdn3mwmrzvh0000gn/T/ipykernel_27095/2245209481.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datafr['Time'] = pd.to_datetime(datafr['Time'])


In [9]:
df_join

,Time Stamp,Date Calc,Time Calc,Subject,Sample,Patient ID,pH,pCO2,pO2,sO2,...,tHb,UPI,K+,Na+,Ca++,Cl-,Glucose,Lactate,p50,cBase
0,2023-08-09 14:59:00,2023-08-09,14:59:00,5,25,5.25,7.425,40.9,37.9,70.6,...,11.9,NaN,3.6,140,1.23,103,5.3,0.5,27.50,2.4
1,2023-08-09 14:58:00,2023-08-09,14:58:00,5,23,5.23,7.425,41.1,37.0,68.5,...,11.9,1136.0,3.7,140,1.22,103,5.1,0.4,27.78,2.6
2,2023-08-09 14:55:00,2023-08-09,14:55:00,5,21,5.21,7.433,39.7,45.1,80.9,...,11.8,NaN,3.6,140,1.23,104,5.1,0.4,26.74,2.2
3,2023-08-09 14:54:00,2023-08-09,14:54:00,5,19,5.19,7.429,40.1,45.8,81.0,...,11.7,1136.0,3.6,140,1.23,104,5.1,0.4,27.12,2.2
4,2023-08-09 14:51:00,2023-08-09,14:51:00,5,17,5.17,7.424,40.3,59.9,90.8,...,11.7,1136.0,3.6,140,1.23,104,5.1,0.4,26.35,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,2023-08-09 09:47:00,2023-08-09,09:47:00,1,10,1.10,7.436,42.5,39.9,74.8,...,11.7,1096.0,4.3,140,1.20,103,4.9,1.0,26.74,4.4
58,2023-08-09 09:44:00,2023-08-09,09:44:00,1,8,1.8,7.433,41.6,46.0,81.6,...,11.5,1096.0,4.1,141,1.17,105,4.6,1.0,26.78,3.5
59,2023-08-09 09:43:00,2023-08-09,09:43:00,1,6,1.6,7.438,42.8,47.3,83.4,...,11.6,1096.0,4.6,140,1.22,103,4.7,1.1,26.30,4.8
60,2023-08-09 09:40:00,2023-08-09,09:40:00,1,4,1.4,7.413,42.8,63.6,92.5,...,11.3,1096.0,4.1,141,1.21,105,4.4,1.0,25.79,2.8


In [10]:
len(pd.unique(df_join["Patient ID"]))

125

In [11]:
pd.unique(df_join['Patient ID'])

array(['5.25', '5.23', '5.21', '5.19', '5.17', '5.15', '5.13', '5.11',
       '5.9', '5.7', '5.5', '5.3', '5.1', '4.25', '4.23', '4.21', '4.19',
       '4.17', '4.15', '4.13', '4.11', '4.9', '4.7', '4.4', '4.2', '3.25',
       '3.23', '3.21', '3.19', '3.17', '3.15', '3.13', '3.11', '3.9',
       '3.7', '3.5', '3.3', '3.1', '2.24', '2.22', '2.20', '2.18', '2.16',
       '2.14', '2.12', '2.10', '2.8', '2.6', '2.4', '2.2', '1.25', '1.23',
       '1.21', '1.19', '1.17', '1.15', '1.13', '1.11', '1.9', '1.7',
       '1.5', '1.3', '1.1', '5.24', '5.22', '5.20', '5.18', '5.16',
       '5.14', '5.12', '5.10', '5.8', '5.6', '5.4', '5.2', '4.24', '4.22',
       '4.20', '4.18', '4.16', '4.14', '4.12', '4.10', '4.8', '4.6',
       '4.5', '4.3', '4.1', '3.24', '3.22', '3.20', '3.18', '3.16',
       '3.14', '3.12', '3.10', '3.8', '3.6', '3.4', '3.2', '2.25', '2.23',
       '2.21', '2.19', '2.17', '2.15', '2.13', '2.11', '2.9', '2.7',
       '2.5', '2.3', '2.1', '1.24', '1.22', '1.20', '1.18', '1.16',